### (1) Imports, paths, and config

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from hydra import compose, initialize
from hydra.utils import instantiate

home_dir = os.environ["HOME"]
bliss_dir = f"{home_dir}/bliss"

os.chdir(f"{bliss_dir}/case_studies/weak_lensing/")
from bliss.catalog import TileCatalog

In [ ]:
with initialize(config_path="../", version_base=None):
    cfg = compose("lensing_config")  # , {"prior.batch_size=4"})

---

### (2) Generate synthetic images

In [ ]:
simulator = instantiate(cfg.simulator)
batch = simulator.get_batch()

In [ ]:
_ = plt.imshow(batch["images"][0][2])

---

### (3) Instantiate encoder

In [ ]:
encoder = instantiate(cfg.train.encoder)
target_cat = TileCatalog(batch["tile_catalog"])
truth_callback = lambda _: target_cat

---

### (4) Train encoder on a single batch of synthetic images

Here we'll just try to learn the shear and convergence for the single batch of synthetic images we generated above. When we actually train via the terminal, we'll do so by pulling in batches from a cached data set. With this single batch, we should be able to infer shear and convergence almost exactly after a sufficient number of iterations.

In [ ]:
num_iters = 2000
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

for i in range(num_iters):
    # Forward pass of encoder
    pred = encoder.infer(batch, truth_callback)
    lvd = pred["marginal"]

    # Compute loss and take optimizer step
    optimizer.zero_grad()
    loss = lvd.compute_nll(target_cat).mean()
    loss.backward()
    optimizer.step()

    if i % 10 == 0:
        print("Iteration {}: Loss {}".format(i, loss.item()))

---

### (5) Summarize results

In [ ]:
# Optimized variational distribution
q = lvd.factors

#### Horizontal shear

In [ ]:
true_shear1_map = batch["tile_catalog"]["shear"].squeeze()[0][:, :, 0]
posterior_mean_shear1_map = q["shear"].mean[0][:, :, 0].detach()

In [ ]:
np.corrcoef(true_shear1_map.flatten(), posterior_mean_shear1_map.flatten())

In [ ]:
fig, (true, posterior) = plt.subplots(nrows=1, ncols=2)
_ = true.imshow(true_shear1_map)
_ = posterior.imshow(posterior_mean_shear1_map)

#### Diagonal shear

In [ ]:
true_shear2_map = batch["tile_catalog"]["shear"].squeeze()[0][:, :, 1]
posterior_mean_shear2_map = q["shear"].mean[0][:, :, 1].detach()

In [ ]:
np.corrcoef(true_shear2_map.flatten(), posterior_mean_shear2_map.flatten())

In [ ]:
fig, (true, posterior) = plt.subplots(nrows=1, ncols=2)
_ = true.imshow(true_shear2_map)
_ = posterior.imshow(posterior_mean_shear2_map)

#### Convergence

In [ ]:
true_convergence_map = batch["tile_catalog"]["convergence"].squeeze()[0]
posterior_mean_convergence_map = q["convergence"].mean[0].detach()

In [ ]:
np.corrcoef(true_convergence_map.flatten(), posterior_mean_convergence_map.flatten())

In [ ]:
fig, (true, posterior) = plt.subplots(nrows=1, ncols=2)
_ = true.imshow(true_convergence_map)
_ = posterior.imshow(posterior_mean_convergence_map)